### SQL Using Python

In [0]:
df_shipped = spark.sql("select * from ecommerce.bronze.orders where order_status = 'shipped'")
display(df_shipped)

# spark.sql("""
#           create or replace table ecommerce.silver.orders 
#           as select * from ecommerce.bronze.orders 
#           where order_status = 'shipped'
#           """).show()

### SQL Using %sql

In [0]:
%sql
select * from ecommerce.bronze.orders where order_status = 'delivered';

-- create or replace table ecommerce.silver.orders 
-- as select * from ecommerce.bronze.orders 
-- where order_status = 'shipped';

### Joins

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1, "Asha", "IN", True),
    (2, "James", "US", False),
    (3, "Leila", "CN", True),
    (4, "Ramesh", "US", None),
    (None, "Chloe", "UK", False)
]

rows_orders = [
    (101, 1, 120.0, "IN"),
    (102, 1, 80.0, "IN"),
    (103, 2, 50.0, "US"),
    (104, 5, 30.0, "DE"),
    (105, 3, 200.0, "CN"),
    (106, None, 15.0, "UK"),
    (107, 3, 40.0, "CN"),
    (108, 2, 75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("vip", T.BooleanType(), True)
])

schema_orders = T.StructType([
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("country", T.StringType(), True)
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders = spark.createDataFrame(rows_orders, schema_orders)

display(df_customers)
display(df_orders)

In [0]:
df_inner = df_orders.join(df_customers, on = "customer_id", how = "inner") # joins wherever the customer_id is the same and present in both tables
df_left = df_orders.join(df_customers, on = "customer_id", how = "left") # keeps all the rows from the left table and only those from the right table where the customer_id is the same
df_right = df_orders.join(df_customers, on = "customer_id", how = "right") # same but with the right table
df_multi = df_orders.join(df_customers, on = ["customer_id", "country"], how = "inner") # joins wherever the customer_id and country are the same and present in both tables

o = df_orders.alias("o")
c = df_customers.alias("c")

df_inner_clean = (
    o.join(c, on="customer_id", how="inner")
    .select(
        "order_id",
        "customer_id",
        "amount",
        F.col("o.country").alias("order_country"),
        "name",
        F.col("c.country").alias("customer_country"),
        "vip"
    )
)

display(df_inner_clean)